In [1]:
"""
Poisson_Ablation.py

Ablation comparing:
1) KAPI Poisson meta-solver trained over a parametric family
2) Single-instance shallow PINN obtained by removing the meta-network

The comparison is performed on one representative fixed task:
    (x0, y0, nu) = (0.50, 0.50, 0.07)

Outputs saved:
- ablation_summary.csv
- ablation_summary.json
- ablation_comparison.png
- ablation_training_curves.png
- ablation_outputs.npz

"""
from __future__ import annotations

import os
import csv
import json
import time
import math
import random
import argparse
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# ------------------------------------------------------------
# Reproducibility / device
# ------------------------------------------------------------
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def get_device(use_gpu: bool) -> torch.device:
    if use_gpu and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


# ------------------------------------------------------------
# Fixed ablation task
# ------------------------------------------------------------
X0_FIXED = 0.50
Y0_FIXED = 0.50
NU_FIXED = 0.07

# KAPI training ranges
X0_MIN, X0_MAX = 0.4, 0.6
Y0_MIN, Y0_MAX = 0.4, 0.6
NU_MIN, NU_MAX = 0.05, 0.10


# ------------------------------------------------------------
# PDE utilities
# ------------------------------------------------------------
def gaussian_source(x: torch.Tensor, y: torch.Tensor, x0, y0, nu) -> torch.Tensor:
    r2 = (x - x0) ** 2 + (y - y0) ** 2
    return (1.0 / (2.0 * torch.pi * nu ** 2)) * torch.exp(-r2 / (2.0 * nu ** 2))


def poisson_exact_fdm(N: int = 60, x0: float = X0_FIXED, y0: float = Y0_FIXED, nu: float = NU_FIXED):
    x = np.linspace(0, 1, N)
    y = np.linspace(0, 1, N)
    dx = x[1] - x[0]
    X, Y = np.meshgrid(x, y, indexing="ij")

    r2 = (X - x0) ** 2 + (Y - y0) ** 2
    f = (1.0 / (2.0 * np.pi * nu ** 2)) * np.exp(-r2 / (2.0 * nu ** 2))

    Nint = N - 2
    f_inner = f[1:-1, 1:-1].reshape(-1)

    e = np.ones(Nint)
    D2 = (np.diag(-2 * e) + np.diag(e[:-1], 1) + np.diag(e[:-1], -1)) / dx ** 2
    I = np.eye(Nint)
    L = np.kron(I, D2) + np.kron(D2, I)
    rhs = -f_inner

    u_inner = np.linalg.solve(L, rhs)
    u = np.zeros((N, N))
    u[1:-1, 1:-1] = u_inner.reshape(Nint, Nint)
    return x, y, u


# ------------------------------------------------------------
# Sampling
# ------------------------------------------------------------
def sample_pde_param(device: torch.device, nu_min_curr: float, nu_max_curr: float) -> torch.Tensor:
    x0 = X0_MIN + (X0_MAX - X0_MIN) * torch.rand(1, device=device)
    y0 = Y0_MIN + (Y0_MAX - Y0_MIN) * torch.rand(1, device=device)

    u = torch.rand(1, device=device)
    nu_min_t = torch.tensor(nu_min_curr, device=device)
    nu_max_t = torch.tensor(nu_max_curr, device=device)
    nu = 10.0 ** (torch.log10(nu_min_t) +
                  (torch.log10(nu_max_t) - torch.log10(nu_min_t)) * u)
    return torch.stack([x0, y0, nu], dim=-1)


def sample_collocation_points_p_dependent(
    N_int: int,
    N_bc: int,
    p: torch.Tensor,
    device: torch.device,
    alpha=None,
    sigma_factor: float = 3.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    x0, y0, nu = p[0]
    if alpha is None:
        alpha = 0.9 if nu.item() < 0.06 else 0.7

    N_loc = int(alpha * N_int)
    N_uni = N_int - N_loc

    if N_loc > 0:
        loc_x = x0 + sigma_factor * nu * torch.randn(N_loc, 1, device=device)
        loc_y = y0 + sigma_factor * nu * torch.randn(N_loc, 1, device=device)
        loc_x = loc_x.clamp(0.0, 1.0)
        loc_y = loc_y.clamp(0.0, 1.0)
        xy_loc = torch.cat([loc_x, loc_y], dim=1)
    else:
        xy_loc = torch.empty(0, 2, device=device)

    if N_uni > 0:
        uni_x = torch.rand(N_uni, 1, device=device)
        uni_y = torch.rand(N_uni, 1, device=device)
        xy_uni = torch.cat([uni_x, uni_y], dim=1)
    else:
        xy_uni = torch.empty(0, 2, device=device)

    xy_int = torch.cat([xy_loc, xy_uni], dim=0)

    if N_bc > 0:
        N_side = N_bc // 4
        t = torch.rand(N_side, 1, device=device)
        xb = torch.cat([
            torch.zeros(N_side, 1, device=device),
            torch.ones(N_side, 1, device=device),
            t,
            t
        ], dim=0)
        yb = torch.cat([
            t,
            t,
            torch.zeros(N_side, 1, device=device),
            torch.ones(N_side, 1, device=device)
        ], dim=0)
        xy_bc = torch.cat([xb, yb], dim=1)
    else:
        xy_bc = torch.empty(0, 2, device=device)

    return xy_int, xy_bc


def sample_collocation_points_fixed_task(
    N_int: int,
    N_bc: int,
    device: torch.device,
    x0: float = X0_FIXED,
    y0: float = Y0_FIXED,
    nu: float = NU_FIXED,
    alpha=None,
    sigma_factor: float = 3.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    p = torch.tensor([[x0, y0, nu]], dtype=torch.float32, device=device)
    return sample_collocation_points_p_dependent(N_int, N_bc, p, device, alpha, sigma_factor)


# ------------------------------------------------------------
# KAPI model
# ------------------------------------------------------------
class KAPIRBF_PINN(nn.Module):
    def __init__(self, M: int = 128, hidden_meta: int = 64):
        super().__init__()
        self.M = M
        self.meta = nn.Sequential(
            nn.Linear(3, hidden_meta),
            nn.Tanh(),
            nn.Linear(hidden_meta, hidden_meta),
            nn.Tanh(),
            nn.Linear(hidden_meta, 4 * M),
        )
        self.c = nn.Parameter(1e-2 * torch.randn(M))

    def _forward_raw(self, p: torch.Tensor, xy: torch.Tensor):
        p_mean = p.new_tensor([0.5, 0.5, 0.075])
        p_std = p.new_tensor([0.1, 0.1, 0.025])
        p_norm = (p - p_mean) / p_std

        meta_out = self.meta(p_norm).view(1, 4, self.M)
        g_logits = meta_out[:, 0, :]
        mu_raw_x = meta_out[:, 1, :]
        mu_raw_y = meta_out[:, 2, :]
        log_sigraw = meta_out[:, 3, :]

        g = torch.sigmoid(g_logits).view(self.M)
        mu_x = torch.sigmoid(mu_raw_x).view(self.M)
        mu_y = torch.sigmoid(mu_raw_y).view(self.M)
        sigma = torch.nn.functional.softplus(log_sigraw).view(self.M) + 5e-4

        x = xy[:, 0:1]
        y = xy[:, 1:2]
        x_diff = x - mu_x.view(1, self.M)
        y_diff = y - mu_y.view(1, self.M)
        r2 = (x_diff ** 2 + y_diff ** 2) / (sigma.view(1, self.M) ** 2)
        phi = torch.exp(-r2)
        u_raw = phi @ (g * self.c)
        return u_raw, (g, mu_x, mu_y, sigma)

    def forward(self, p: torch.Tensor, xy: torch.Tensor):
        u_raw, aux = self._forward_raw(p, xy)
        x, y = xy[:, 0:1], xy[:, 1:2]
        bc_factor = x * (1.0 - x) * y * (1.0 - y)
        u = u_raw * bc_factor.view(-1)
        return u, aux


# ------------------------------------------------------------
# Single-instance PINN (no meta-network)
# ------------------------------------------------------------
class SingleTaskPoissonPINN(nn.Module):
    def __init__(self, M: int = 128):
        super().__init__()
        self.M = M
        self.g_logits = nn.Parameter(torch.zeros(M))
        self.mu_x_raw = nn.Parameter(torch.randn(M) * 0.25)
        self.mu_y_raw = nn.Parameter(torch.randn(M) * 0.25)
        self.log_sig_raw = nn.Parameter(torch.randn(M) * 0.10)
        self.c = nn.Parameter(1e-2 * torch.randn(M))

    def _forward_raw(self, xy: torch.Tensor):
        x = xy[:, 0:1]
        y = xy[:, 1:2]
        g = torch.sigmoid(self.g_logits)
        mu_x = torch.sigmoid(self.mu_x_raw)
        mu_y = torch.sigmoid(self.mu_y_raw)
        sigma = torch.nn.functional.softplus(self.log_sig_raw) + 5e-4
        x_diff = x - mu_x.view(1, self.M)
        y_diff = y - mu_y.view(1, self.M)
        r2 = (x_diff ** 2 + y_diff ** 2) / (sigma.view(1, self.M) ** 2)
        phi = torch.exp(-r2)
        u_raw = phi @ (g * self.c)
        return u_raw, (g, mu_x, mu_y, sigma)

    def forward(self, xy: torch.Tensor):
        u_raw, aux = self._forward_raw(xy)
        x, y = xy[:, 0:1], xy[:, 1:2]
        bc_factor = x * (1.0 - x) * y * (1.0 - y)
        u = u_raw * bc_factor.view(-1)
        return u, aux


# ------------------------------------------------------------
# Residuals
# ------------------------------------------------------------
def poisson_residual_kapi(model: KAPIRBF_PINN, p: torch.Tensor, xy_int: torch.Tensor) -> torch.Tensor:
    xy_int = xy_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xy_int)

    grads = torch.autograd.grad(u, xy_int, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = grads[:, 0], grads[:, 1]
    grads2_x = torch.autograd.grad(u_x, xy_int, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    grads2_y = torch.autograd.grad(u_y, xy_int, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_xx, u_yy = grads2_x[:, 0], grads2_y[:, 1]
    x, y = xy_int[:, 0], xy_int[:, 1]
    x0, y0, nu = p[0, 0], p[0, 1], p[0, 2]
    f_val = gaussian_source(x, y, x0, y0, nu)
    return -(u_xx + u_yy) - f_val


def poisson_residual_single(model: SingleTaskPoissonPINN, xy_int: torch.Tensor, x0, y0, nu) -> torch.Tensor:
    xy_int = xy_int.clone().detach().requires_grad_(True)
    u, _ = model(xy_int)

    grads = torch.autograd.grad(u, xy_int, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = grads[:, 0], grads[:, 1]
    grads2_x = torch.autograd.grad(u_x, xy_int, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    grads2_y = torch.autograd.grad(u_y, xy_int, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_xx, u_yy = grads2_x[:, 0], grads2_y[:, 1]
    x, y = xy_int[:, 0], xy_int[:, 1]
    f_val = gaussian_source(x, y, x0, y0, nu)
    return -(u_xx + u_yy) - f_val


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
def train_kapi_poisson(
    device: torch.device,
    n_int: int = 2048,
    n_bc: int = 256,
    epochs: int = 2000,
    lr: float = 1e-3,
    M: int = 128,
    tasks_per_batch: int = 4,
):
    model = KAPIRBF_PINN(M=M, hidden_meta=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

    best_loss = float("inf")
    best_state = None
    history = {"epoch": [], "loss": [], "loss_pde": [], "loss_bc": [], "lr": []}

    t0 = time.perf_counter()
    for ep in range(1, epochs + 1):
        optimizer.zero_grad()
        total_loss = 0.0
        total_pde = 0.0
        total_bc = 0.0

        if ep <= epochs // 2:
            nu_min_curr, nu_max_curr = 0.08, NU_MAX
        else:
            nu_min_curr, nu_max_curr = NU_MIN, NU_MAX

        for _ in range(tasks_per_batch):
            p = sample_pde_param(device, nu_min_curr, nu_max_curr)
            xy_int, xy_bc = sample_collocation_points_p_dependent(n_int, n_bc, p, device)
            res_int = poisson_residual_kapi(model, p, xy_int)
            loss_pde = torch.mean(res_int ** 2)

            if xy_bc.shape[0] > 0:
                u_bc, (g_bc, _, _, _) = model(p, xy_bc)
                loss_bc = torch.mean(u_bc ** 2)
            else:
                _, (g_bc, _, _, _) = model(p, torch.zeros(1, 2, device=device))
                loss_bc = torch.tensor(0.0, device=device)

            loss_sparse = 1e-5 * torch.mean(torch.abs(g_bc))
            loss_task = loss_pde + loss_sparse
            total_loss += loss_task
            total_pde += loss_pde
            total_bc += loss_bc

        total_loss /= tasks_per_batch
        total_pde /= tasks_per_batch
        total_bc /= tasks_per_batch

        total_loss.backward()
        optimizer.step()
        scheduler.step()

        if total_loss.item() < best_loss:
            best_loss = total_loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history["epoch"].append(ep)
        history["loss"].append(float(total_loss.item()))
        history["loss_pde"].append(float(total_pde.item()))
        history["loss_bc"].append(float(total_bc.item()))
        history["lr"].append(float(scheduler.get_last_lr()[0]))

        if ep % 200 == 0 or ep == 1:
            print(f"[KAPI] Epoch {ep:5d} | Loss {total_loss.item():.3e} | PDE {total_pde.item():.3e}")

    train_time = time.perf_counter() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, train_time


def train_single_poisson(
    device: torch.device,
    n_int: int = 2048,
    n_bc: int = 256,
    epochs: int = 2000,
    lr: float = 1e-3,
    M: int = 128,
):
    model = SingleTaskPoissonPINN(M=M).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

    x0 = torch.tensor(X0_FIXED, device=device)
    y0 = torch.tensor(Y0_FIXED, device=device)
    nu = torch.tensor(NU_FIXED, device=device)

    best_loss = float("inf")
    best_state = None
    history = {"epoch": [], "loss": [], "loss_pde": [], "loss_bc": [], "lr": []}

    t0 = time.perf_counter()
    for ep in range(1, epochs + 1):
        optimizer.zero_grad()
        xy_int, xy_bc = sample_collocation_points_fixed_task(n_int, n_bc, device)
        res_int = poisson_residual_single(model, xy_int, x0, y0, nu)
        loss_pde = torch.mean(res_int ** 2)

        if xy_bc.shape[0] > 0:
            u_bc, (g_bc, _, _, _) = model(xy_bc)
            loss_bc = torch.mean(u_bc ** 2)
        else:
            _, (g_bc, _, _, _) = model(torch.zeros(1, 2, device=device))
            loss_bc = torch.tensor(0.0, device=device)

        loss_sparse = 1e-5 * torch.mean(torch.abs(g_bc))
        loss = loss_pde + loss_sparse
        loss.backward()
        optimizer.step()
        scheduler.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history["epoch"].append(ep)
        history["loss"].append(float(loss.item()))
        history["loss_pde"].append(float(loss_pde.item()))
        history["loss_bc"].append(float(loss_bc.item()))
        history["lr"].append(float(scheduler.get_last_lr()[0]))

        if ep % 200 == 0 or ep == 1:
            print(f"[PINN] Epoch {ep:5d} | Loss {loss.item():.3e} | PDE {loss_pde.item():.3e}")

    train_time = time.perf_counter() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, train_time


# ------------------------------------------------------------
# Evaluation / plots / outputs
# ------------------------------------------------------------
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_kapi(model: KAPIRBF_PINN, device: torch.device, N: int = 60) -> Dict[str, np.ndarray]:
    xg, yg, u_exact = poisson_exact_fdm(N=N, x0=X0_FIXED, y0=Y0_FIXED, nu=NU_FIXED)
    Xg, Yg = np.meshgrid(xg, yg, indexing="ij")
    xy_grid = np.stack([Xg.reshape(-1), Yg.reshape(-1)], axis=1)
    xy_grid_t = torch.tensor(xy_grid, dtype=torch.float32, device=device)
    p_test = torch.tensor([[X0_FIXED, Y0_FIXED, NU_FIXED]], dtype=torch.float32, device=device)

    t0 = time.perf_counter()
    model.eval()
    with torch.no_grad():
        u_pred_flat, (g, mu_x, mu_y, sigma) = model(p_test, xy_grid_t)
    infer_time = time.perf_counter() - t0

    u_pred = u_pred_flat.cpu().numpy().reshape(N, N)
    err = u_pred - u_exact
    abs_err = np.abs(err)
    rel_l2 = np.linalg.norm(err.ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)
    return dict(
        x=xg, y=yg, u_exact=u_exact, u_pred=u_pred, abs_error=abs_err, rel_l2=rel_l2,
        infer_time=infer_time, mu_x=mu_x.cpu().numpy(), mu_y=mu_y.cpu().numpy(),
        sigma=sigma.cpu().numpy(), gate=g.cpu().numpy()
    )


def evaluate_single(model: SingleTaskPoissonPINN, device: torch.device, N: int = 60) -> Dict[str, np.ndarray]:
    xg, yg, u_exact = poisson_exact_fdm(N=N, x0=X0_FIXED, y0=Y0_FIXED, nu=NU_FIXED)
    Xg, Yg = np.meshgrid(xg, yg, indexing="ij")
    xy_grid = np.stack([Xg.reshape(-1), Yg.reshape(-1)], axis=1)
    xy_grid_t = torch.tensor(xy_grid, dtype=torch.float32, device=device)

    t0 = time.perf_counter()
    model.eval()
    with torch.no_grad():
        u_pred_flat, (g, mu_x, mu_y, sigma) = model(xy_grid_t)
    infer_time = time.perf_counter() - t0

    u_pred = u_pred_flat.cpu().numpy().reshape(N, N)
    err = u_pred - u_exact
    abs_err = np.abs(err)
    rel_l2 = np.linalg.norm(err.ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)
    return dict(
        x=xg, y=yg, u_exact=u_exact, u_pred=u_pred, abs_error=abs_err, rel_l2=rel_l2,
        infer_time=infer_time, mu_x=mu_x.cpu().numpy(), mu_y=mu_y.cpu().numpy(),
        sigma=sigma.cpu().numpy(), gate=g.cpu().numpy()
    )


def save_summary(out_dir: Path, kapi_history, pinn_history, kapi_eval, pinn_eval, kapi_train_time, pinn_train_time, kapi_params, pinn_params):
    rows = [
        {
            "method": "KAPI_meta_solver",
            "rel_l2_error": float(kapi_eval["rel_l2"]),
            "offline_train_time_sec": float(kapi_train_time),
            "test_time_inference_sec": float(kapi_eval["infer_time"]),
            "trainable_params": int(kapi_params),
            "requires_retraining_per_new_task": "No",
        },
        {
            "method": "Single_instance_PINN",
            "rel_l2_error": float(pinn_eval["rel_l2"]),
            "offline_train_time_sec": float(pinn_train_time),
            "test_time_inference_sec": float(pinn_eval["infer_time"]),
            "trainable_params": int(pinn_params),
            "requires_retraining_per_new_task": "Yes",
        },
    ]

    csv_path = out_dir / "ablation_summary.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    json_path = out_dir / "ablation_summary.json"
    payload = {
        "fixed_task": {"x0": X0_FIXED, "y0": Y0_FIXED, "nu": NU_FIXED},
        "kapi": rows[0],
        "single_instance_pinn": rows[1],
        "notes": "KAPI reports offline family-training time plus near-zero test-time inference; single-instance PINN reports per-task training time."
    }
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2)

    np.savez_compressed(
        out_dir / "ablation_outputs.npz",
        x=kapi_eval["x"].astype(np.float32),
        y=kapi_eval["y"].astype(np.float32),
        u_exact=kapi_eval["u_exact"].astype(np.float32),
        u_kapi=kapi_eval["u_pred"].astype(np.float32),
        e_kapi=kapi_eval["abs_error"].astype(np.float32),
        u_pinn=pinn_eval["u_pred"].astype(np.float32),
        e_pinn=pinn_eval["abs_error"].astype(np.float32),
        kapi_rel_l2=np.float32(kapi_eval["rel_l2"]),
        pinn_rel_l2=np.float32(pinn_eval["rel_l2"]),
        kapi_train_time=np.float32(kapi_train_time),
        pinn_train_time=np.float32(pinn_train_time),
        kapi_infer_time=np.float32(kapi_eval["infer_time"]),
        pinn_infer_time=np.float32(pinn_eval["infer_time"]),
    )


def make_plots(out_dir: Path, kapi_history, pinn_history, kapi_eval, pinn_eval):
    # Training curves
    plt.figure(figsize=(7.4, 4.6))
    plt.plot(kapi_history["epoch"], kapi_history["loss"], lw=2, label="KAPI total")
    plt.plot(pinn_history["epoch"], pinn_history["loss"], lw=2, label="Single-instance PINN total")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Poisson ablation: training curves")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "ablation_training_curves.png", dpi=300)
    plt.close()

    # Comparison figure
    u_exact = kapi_eval["u_exact"]
    u_kapi = kapi_eval["u_pred"]
    e_kapi = kapi_eval["abs_error"]
    u_pinn = pinn_eval["u_pred"]
    e_pinn = pinn_eval["abs_error"]

    sol_vmin = min(np.min(u_exact), np.min(u_kapi), np.min(u_pinn))
    sol_vmax = max(np.max(u_exact), np.max(u_kapi), np.max(u_pinn))
    err_vmax = max(np.max(e_kapi), np.max(e_pinn))

    fig, axs = plt.subplots(2, 3, figsize=(12.5, 7.4), constrained_layout=True)
    ims = []
    ims.append(axs[0,0].imshow(u_exact.T, origin="lower", extent=[0,1,0,1], vmin=sol_vmin, vmax=sol_vmax))
    axs[0,0].set_title(r"Exact (FDM)")
    ims.append(axs[0,1].imshow(u_kapi.T, origin="lower", extent=[0,1,0,1], vmin=sol_vmin, vmax=sol_vmax))
    axs[0,1].set_title(r"KAPI")
    ims.append(axs[0,2].imshow(u_pinn.T, origin="lower", extent=[0,1,0,1], vmin=sol_vmin, vmax=sol_vmax))
    axs[0,2].set_title(r"Single-instance PINN")
    #ims.append(axs[1,0].imshow(np.zeros_like(u_exact).T, origin="lower", extent=[0,1,0,1], vmin=0, vmax=1))#--->
    axs[1,0].axis("off")
    ims.append(axs[1,1].imshow(e_kapi.T, origin="lower", extent=[0,1,0,1], vmin=0.0, vmax=err_vmax))
    axs[1,1].set_title(r"$|u_{\mathrm{KAPI}}-u_{\mathrm{exact}}|$")
    ims.append(axs[1,2].imshow(e_pinn.T, origin="lower", extent=[0,1,0,1], vmin=0.0, vmax=err_vmax))
    axs[1,2].set_title(r"$|u_{\mathrm{PINN}}-u_{\mathrm{exact}}|$")

    for ax in [axs[0, 0], axs[0, 1], axs[0, 2], axs[1, 1], axs[1, 2]]:
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    cbar1 = fig.colorbar(ims[2], ax=axs[0, :], shrink=0.9)
    cbar1.set_label(r"$u(x,y)$")
    cbar2 = fig.colorbar(ims[4], ax=axs[1, 1:], shrink=0.9)
    cbar2.set_label("Absolute Error")

    fig.suptitle(rf"Poisson ablation on fixed task $(x_0,y_0,\nu)=({X0_FIXED:.2f},{Y0_FIXED:.2f},{NU_FIXED:.2f})$")
    fig.savefig(out_dir / "ablation_comparison.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(description="Poisson ablation: KAPI vs single-instance PINN")
    parser.add_argument("--gpu", action="store_true", help="Use GPU if available")
    parser.add_argument("--outdir", type=str, default="poisson_ablation_run")
    parser.add_argument("--epochs_kapi", type=int, default=2000)
    parser.add_argument("--epochs_pinn", type=int, default=2000)
    parser.add_argument("--lr_kapi", type=float, default=1e-3)
    parser.add_argument("--lr_pinn", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=128)
    parser.add_argument("--n_int", type=int, default=2048)
    parser.add_argument("--n_bc", type=int, default=256)
    parser.add_argument("--tasks_per_batch", type=int, default=4)
    parser.add_argument("--eval_N", type=int, default=60)
    args, _unknown = parser.parse_known_args()

    device = get_device(args.gpu)
    out_dir = Path(args.outdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Using device:", device)
    print(f"Running ablation on fixed task (x0, y0, nu)=({X0_FIXED}, {Y0_FIXED}, {NU_FIXED})")

    kapi_model, kapi_history, kapi_train_time = train_kapi_poisson(
        device=device,
        n_int=args.n_int,
        n_bc=args.n_bc,
        epochs=args.epochs_kapi,
        lr=args.lr_kapi,
        M=args.M,
        tasks_per_batch=args.tasks_per_batch,
    )
    pinn_model, pinn_history, pinn_train_time = train_single_poisson(
        device=device,
        n_int=args.n_int,
        n_bc=args.n_bc,
        epochs=args.epochs_pinn,
        lr=args.lr_pinn,
        M=args.M,
    )

    torch.save(kapi_model.state_dict(), out_dir / "kapi_model.pt")
    torch.save(pinn_model.state_dict(), out_dir / "single_instance_pinn_model.pt")

    kapi_eval = evaluate_kapi(kapi_model, device=device, N=args.eval_N)
    pinn_eval = evaluate_single(pinn_model, device=device, N=args.eval_N)

    kapi_params = count_params(kapi_model)
    pinn_params = count_params(pinn_model)

    save_summary(out_dir, kapi_history, pinn_history, kapi_eval, pinn_eval,
                 kapi_train_time, pinn_train_time, kapi_params, pinn_params)
    make_plots(out_dir, kapi_history, pinn_history, kapi_eval, pinn_eval)

    print("\nSummary:")
    print(f"  KAPI relL2       : {kapi_eval['rel_l2']:.3e}")
    print(f"  PINN relL2       : {pinn_eval['rel_l2']:.3e}")
    print(f"  KAPI train time  : {kapi_train_time:.2f} s")
    print(f"  PINN train time  : {pinn_train_time:.2f} s")
    print(f"  KAPI inference   : {kapi_eval['infer_time']:.6f} s")
    print(f"  PINN inference   : {pinn_eval['infer_time']:.6f} s")
    print(f"  KAPI params      : {kapi_params}")
    print(f"  PINN params      : {pinn_params}")
    print(f"\nSaved outputs in: {out_dir}")


if __name__ == "__main__":
    main()
    


Using device: cpu
Running ablation on fixed task (x0, y0, nu)=(0.5, 0.5, 0.07)
[KAPI] Epoch     1 | Loss 2.038e+01 | PDE 2.038e+01
[KAPI] Epoch   200 | Loss 8.349e-02 | PDE 8.348e-02
[KAPI] Epoch   400 | Loss 2.830e-02 | PDE 2.830e-02
[KAPI] Epoch   600 | Loss 2.843e-02 | PDE 2.842e-02
[KAPI] Epoch   800 | Loss 4.900e-02 | PDE 4.899e-02
[KAPI] Epoch  1000 | Loss 1.882e-02 | PDE 1.882e-02
[KAPI] Epoch  1200 | Loss 1.212e-01 | PDE 1.212e-01
[KAPI] Epoch  1400 | Loss 6.843e-02 | PDE 6.842e-02
[KAPI] Epoch  1600 | Loss 4.818e-02 | PDE 4.817e-02
[KAPI] Epoch  1800 | Loss 5.720e-02 | PDE 5.719e-02
[KAPI] Epoch  2000 | Loss 5.148e-02 | PDE 5.147e-02
[PINN] Epoch     1 | Loss 4.542e+01 | PDE 4.542e+01
[PINN] Epoch   200 | Loss 2.371e+01 | PDE 2.371e+01
[PINN] Epoch   400 | Loss 1.614e+01 | PDE 1.614e+01
[PINN] Epoch   600 | Loss 1.085e+01 | PDE 1.085e+01
[PINN] Epoch   800 | Loss 3.527e+00 | PDE 3.527e+00
[PINN] Epoch  1000 | Loss 4.406e-01 | PDE 4.406e-01
[PINN] Epoch  1200 | Loss 2.353e-01 |